# Extension 06 Inverse form finding

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**Given a target equilibrium shape $(x^*, y^*, z^*)$, which force densities $q$ hold it in balance?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** all node coordinates $(x^*, y^*, z^*)$ of a target shell and the loads ${\color{#2e9e4a} (p_x, p_y, p_z)}$ on its 45 ${\color{#d62828} \text{free}}$ nodes.

**Unknown:** the 84 bar force densities $q$.

**Inverse Equilibrium:** rewriting $C_N^{T}\,Q\,(C\,x^*) = C_N^{T}\,\operatorname{diag}(C\,x^*)\,q$ turns equilibrium at fixed geometry into one linear system for $q$:

$$ \begin{bmatrix} C_N^{T}\,\operatorname{diag}(C\,x^*) \\ C_N^{T}\,\operatorname{diag}(C\,y^*) \\ C_N^{T}\,\operatorname{diag}(C\,z^*) \end{bmatrix} q = \begin{bmatrix} {\color{#2e9e4a} p_x} \\ {\color{#2e9e4a} p_y} \\ {\color{#2e9e4a} p_z} \end{bmatrix}, \qquad E\,q = {\color{#2e9e4a} p} $$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_projection_all_axes.svg"
       style="width:100%; height:auto;"
       alt="Equilibrium equations projected along x y and z axes at known geometry" />
  <p style="text-align:center;">
    <em>When node positions are known, the three axis equilibrium equations become a linear system directly in q.</em>
  </p>
</div>

**References:**

- **Ariadne**, https://github.com/adam-t-burke/Ariadne

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Target Vault Geometry

In [1]:
import numpy as np

# nodes and bars of the 6 x 6 grid
n = 7
x, y = np.meshgrid(range(n), range(n))
x = x.flatten().astype(float)
y = y.flatten().astype(float)
z = np.zeros(n * n)

edges = [(k, k+1) for k in range(n*n) if x[k] < n-1] + \
        [(k, k+n) for k in range(n*n) if y[k] < n-1]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

fixed = [0, n-1, n*(n-1), n*n - 1]
free  = [k for k in range(n*n) if k not in fixed]
C_N, C_F = C[:, free], C[:, fixed]

# generate a target compression shell (x_star, y_star, z_star) with unknown reference q_true
q_true = -1.5 * np.ones(len(edges))
for e, (a, b) in enumerate(edges):
    if (x[a] == x[b] and x[a] in (0, n-1)) or (y[a] == y[b] and y[a] in (0, n-1)):
        q_true[e] = -5.0
Q_true = np.diag(q_true)

p_x = np.zeros(len(free))
p_y = np.zeros(len(free))
p_z = -0.3 * np.ones(len(free))

# equation 6.32
A_true = C_N.T @ Q_true @ C_N
x_star, y_star, z_star = x.copy(), y.copy(), z.copy()
x_star[free] = np.linalg.solve(A_true, p_x - C_N.T @ Q_true @ C_F @ x_star[fixed])
y_star[free] = np.linalg.solve(A_true, p_y - C_N.T @ Q_true @ C_F @ y_star[fixed])
z_star[free] = np.linalg.solve(A_true, p_z - C_N.T @ Q_true @ C_F @ z_star[fixed])

# Linear Inverse Solve $E\,q = p$

Assemble the $135 \times 84$ equilibrium matrix $E$ from $(x^*, y^*, z^*)$, solve for $q$ in one least squares line, and plug $q$ back into `# equation 6.32` to verify that it reproduces the target shape.

In [2]:
# assemble E q = p from the target coordinates
E = np.vstack([
    C_N.T @ np.diag(C @ x_star),
    C_N.T @ np.diag(C @ y_star),
    C_N.T @ np.diag(C @ z_star),
])
p = np.concatenate([p_x, p_y, p_z])

# solve for the unknown force densities q
q, _, rank, _ = np.linalg.lstsq(E, p, rcond=None)
Q = np.diag(q)

# check by running forward form finding (# equation 6.32) with the recovered q
A = C_N.T @ Q @ C_N
x_rec, y_rec, z_rec = x.copy(), y.copy(), z.copy()
x_rec[free] = np.linalg.solve(A, p_x - C_N.T @ Q @ C_F @ x_rec[fixed])
y_rec[free] = np.linalg.solve(A, p_y - C_N.T @ Q @ C_F @ y_rec[fixed])
z_rec[free] = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z_rec[fixed])

shape_err = np.max(np.sqrt((x_rec - x_star)**2 + (y_rec - y_star)**2 + (z_rec - z_star)**2))
print("E shape =", E.shape, " rank =", rank)
print("max |q - q_true|     =", f"{np.abs(q - q_true).max():.2e}", "kN/m")
print("forward shape error  =", f"{shape_err:.2e}", "m")

E shape = (135, 84)  rank = 84
max |q - q_true|     = 1.95e-14 kN/m
forward shape error  = 4.00e-15 m


# Viewer

The recovered compression shell $(x_{\text{rec}}, y_{\text{rec}}, z_{\text{rec}})$, shaded by bar force $|f| = |q|\,l$.

In [3]:
from session_py import Color, Matrix, Session
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue
RED  = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)

f_abs = np.abs(q) * np.sqrt((C @ x_rec)**2 + (C @ y_rec)**2 + (C @ z_rec)**2)
colors = [Color(0.65 - 0.65 * fv / f_abs.max(), 0.65 - 0.65 * fv / f_abs.max(), 0.65 - 0.65 * fv / f_abs.max(), 1.0) for fv in f_abs]

session = Session("fdm_ext06_inverse_vault")
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x_rec, y_rec, z_rec]).tolist()), colors, [3.0] * len(edges))

for k in range(n * n):
    pt = session.objects.points[k]
    pt.pointcolor = BLUE if k in fixed else RED
    pt.width = 14.0 if k in fixed else 7.0

show(session, "fdm_ext06_inverse_vault")

# Observation

 Equilibrium is bilinear in $q$ and $x$: fixing $q$ gives the linear forward system $A\,x_N = b$, while fixing $x^*$ gives the linear inverse system $E\,q = p$. When an arbitrary surface is not yet funicular, $E\,q \approx p$ measures its closest static equilibrium state.